# Claimity Partner API - Insurer v1 (Python Notebook)

This notebook shows how to call the **Claimity Partner API** as an **Insurer** organization.

It demonstrates:
- **OAuth 2.0 Client Credentials** using a **JWT client assertion** (RS256)
- **DPoP** proof-of-possession per request (ES256)
- Example calls for the **Insurer v1** endpoints

> Note: This notebook is intended for **production** (`https://app.claimity.ch`).


## How to run

### Prerequisites

- Python **3.10+**
- Internet access to `https://app.claimity.ch`
- Your **Claimity Partner API credentials** (Insurer organization)

### Required files

- A private key file in **PEM (PKCS#8)** format, e.g. `private-key.pem`
  - The corresponding **public key** must be registered in Claimity for your client.

### Install dependencies

```bash
python -m venv .venv
source .venv/bin/activate  # Windows (PowerShell): .venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

### Configure environment variables

Use `docs/openapi/.env.example` as a template.


## Configuration

| Variable | Required | Example | Description |
|---|---:|---|---|
| `CLIENT_ID` | yes | `org-inso-00003` | Your Claimity client id (Insurer org) |
| `PRIVATE_KEY_PATH` | yes | `./private-key-ins.pem` | Path to your **private** RSA key (PEM, PKCS#8) |
| `KID` | sometimes | `my-key-2025-01` | Key id registered with Claimity (if applicable) |
| `TOKEN_URL` | yes | `https://app.claimity.ch/v1/oauth/token` | Claimity OAuth token endpoint |
| `TOKEN_AUDIENCE` | yes | `https://app.claimity.ch/realms/claimity/protocol/openid-connect/token` | Audience for the client assertion (`aud`) |
| `API_BASE_URL` | yes | `https://app.claimity.ch` | Base URL for Partner API calls (`/v1/...`) |

Optional IDs (if you want to skip discovery):

| Variable | Required | Example |
|---|---:|---|
| `CLAIM_ID` | no | `00000000-0000-0000-0000-000000000000` |
| `REPORT_ID` | no | `00000000-0000-0000-0000-000000000000` |
| `DOC_ID` | no | `00000000-0000-0000-0000-000000000000` |


## Security notes (production)

- Do not commit private keys or tokens.
- This notebook clears cell outputs when distributed.
- Store secrets in a secure location (secret manager) and use least privilege.


## Troubleshooting

### 401 invalid_dpop

- `htu` must match the exact URL including query string.
- `iat` must be within the allowed time window (ensure correct system clock).
- new `jti` per request.

### Token request fails

- Verify `TOKEN_URL` and `TOKEN_AUDIENCE`.
- Ensure `PRIVATE_KEY_PATH` matches the registered public key.

### 403 forbidden

- Missing role/organization context.

### 429 too_many_requests

Apply backoff and respect `Retry-After` if present.


In [ ]:
# Standard library
import base64
import hashlib
import json
import os
import time
import uuid
from urllib.parse import urlencode, urljoin

import jwt  # PyJWT
import requests
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import serialization


In [ ]:
# ---- Configuration ----

import os

CLIENT_ID = os.getenv("CLIENT_ID", "org-inso-00003")
SCOPE = os.getenv("SCOPE", "roles")
KID = os.getenv("KID")

TOKEN_REQUEST_URL = os.getenv("TOKEN_URL", "https://app.claimity.ch/v1/oauth/token")
TOKEN_AUDIENCE = os.getenv(
    "TOKEN_AUDIENCE",
    "https://app.claimity.ch/realms/claimity/protocol/openid-connect/token",
)

PRIVATE_KEY_PATH = os.getenv("PRIVATE_KEY_PATH", "./private-key-ins.pem")
API_BASE_URL = os.getenv("API_BASE_URL", "https://app.claimity.ch")

DEBUG_PRINT_TOKEN_RESPONSE = os.getenv("DEBUG_PRINT_TOKEN_RESPONSE", "false").lower() == "true"
DEBUG_PRINT_JWT_CLAIMS = os.getenv("DEBUG_PRINT_JWT_CLAIMS", "false").lower() == "true"

print("CLIENT_ID:", CLIENT_ID)
print("TOKEN_REQUEST_URL:", TOKEN_REQUEST_URL)
print("TOKEN_AUDIENCE:", TOKEN_AUDIENCE)
print("PRIVATE_KEY_PATH:", PRIVATE_KEY_PATH)
print("API_BASE_URL:", API_BASE_URL)


## Authentication (OAuth2 client_credentials + JWT client assertion)


In [ ]:
CLIENT_ASSERTION_TYPE = 'urn:ietf:params:oauth:client-assertion-type:jwt-bearer'

def b64url(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode('ascii').rstrip('=')

def b64url_decode_to_json(segment: str) -> dict:
    s = segment + '=' * (-len(segment) % 4)
    raw = base64.urlsafe_b64decode(s.encode('utf-8'))
    try:
        return json.loads(raw.decode('utf-8'))
    except Exception:
        return {}

def load_private_key_pem(path: str) -> str:
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

def make_client_assertion(client_id: str, audience: str, private_key_pem: str, kid: str | None) -> str:
    headers = {'alg': 'RS256', 'typ': 'JWT'}
    if kid:
        headers['kid'] = kid
    now = int(time.time())
    payload = {
        'iss': client_id,
        'sub': client_id,
        'aud': audience,
        'jti': str(uuid.uuid4()),
        'exp': now + 90,
        'iat': now,
    }
    return jwt.encode(payload, private_key_pem, algorithm='RS256', headers=headers)

def request_access_token(token_request_url: str, client_id: str, assertion: str, scope: str | None = None, timeout: int = 15) -> tuple[int, dict]:
    data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_assertion_type': CLIENT_ASSERTION_TYPE,
        'client_assertion': assertion,
    }
    if scope:
        data['scope'] = scope
    resp = requests.post(token_request_url, data=data, headers={'Content-Type': 'application/x-www-form-urlencoded'}, timeout=timeout)
    try:
        body = resp.json()
    except Exception:
        body = {'raw': resp.text}
    return resp.status_code, body


# ---- Fetch access token ----

def redact_token_response(body: dict) -> dict:
    if not isinstance(body, dict):
        return {"raw": str(body)}
    redacted = dict(body)
    if "access_token" in redacted:
        redacted["access_token"] = "<redacted>"
    if "refresh_token" in redacted:
        redacted["refresh_token"] = "<redacted>"
    return redacted


priv_pem = load_private_key_pem(PRIVATE_KEY_PATH)
assertion = make_client_assertion(CLIENT_ID, TOKEN_AUDIENCE, priv_pem, KID)
status, token_body = request_access_token(TOKEN_REQUEST_URL, CLIENT_ID, assertion, scope=SCOPE)

print("HTTP", status)
if DEBUG_PRINT_TOKEN_RESPONSE:
    print(json.dumps(redact_token_response(token_body), indent=2, ensure_ascii=False))
else:
    if isinstance(token_body, dict):
        print({k: token_body.get(k) for k in ("token_type", "expires_in", "scope") if k in token_body})

if status != 200 or "access_token" not in (token_body or {}):
    print(json.dumps(redact_token_response(token_body), indent=2, ensure_ascii=False))
    raise RuntimeError("Token request failed; see output above.")

ACCESS_TOKEN = token_body["access_token"]
print("Access token received (length):", len(ACCESS_TOKEN))


In [ ]:
# Optional: decode token header/payload (WITHOUT signature verification)

if not DEBUG_PRINT_JWT_CLAIMS:
    print("JWT claim printing disabled. Set DEBUG_PRINT_JWT_CLAIMS=true to enable.")
else:
    parts = ACCESS_TOKEN.split(".")
    if len(parts) == 3:
        header = b64url_decode_to_json(parts[0])
        payload = b64url_decode_to_json(parts[1])
        print("Header:")
        print(json.dumps(header, indent=2, ensure_ascii=False))
        print("Payload:")
        print(json.dumps(payload, indent=2, ensure_ascii=False))
    else:
        print("Unexpected JWT format")


## DPoP (Proof-of-Possession)


In [ ]:
# ---- DPoP helpers ----
def gen_ephemeral_ec_p256_key():
    return ec.generate_private_key(ec.SECP256R1())

def ec_public_jwk_p256(priv_key) -> dict:
    pub = priv_key.public_key()
    numbers = pub.public_numbers()
    x = numbers.x.to_bytes(32, 'big')
    y = numbers.y.to_bytes(32, 'big')
    return {
        'kty': 'EC',
        'crv': 'P-256',
        'x': b64url(x),
        'y': b64url(y),
    }

def dpop_proof(priv_key, method: str, url: str, access_token: str | None) -> str:
    jwk_pub = ec_public_jwk_p256(priv_key)
    headers = {
        'typ': 'dpop+jwt',
        'alg': 'ES256',
        'jwk': jwk_pub,
    }
    now = int(time.time())
    payload = {
        'htm': method.upper(),
        'htu': url,
        'iat': now,
        'jti': str(uuid.uuid4()),
    }
    if access_token:
        ath = hashlib.sha256(access_token.encode('utf-8')).digest()
        payload['ath'] = b64url(ath)

    pem = priv_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    )
    return jwt.encode(payload, pem, algorithm='ES256', headers=headers)

def build_url(base_url: str, path: str, params: dict | None = None) -> str:
    base = base_url.rstrip('/') + '/'
    full = urljoin(base, path.lstrip('/'))
    if params:
        qs = urlencode(params, doseq=True)
        full = f'{full}?{qs}'
    return full

def send_with_dpop(method: str, base_url: str, path: str, access_token: str, params: dict | None = None, payload_json: dict | None = None, timeout: int = 20):
    url = build_url(base_url, path, params)
    dpop_key = gen_ephemeral_ec_p256_key()
    proof = dpop_proof(dpop_key, method, url, access_token)

    headers = {
        'Authorization': f'DPoP {access_token}',
        'DPoP': proof,
        'Accept': 'application/json',
    }
    if payload_json is not None:
        headers['Content-Type'] = 'application/json'
        data = json.dumps(payload_json)
    else:
        data = None

    resp = requests.request(method=method.upper(), url=url, headers=headers, data=data, timeout=timeout)
    ctype = resp.headers.get('content-type', '')
    if 'application/json' in ctype.lower():
        try:
            body = resp.json()
        except Exception:
            body = resp.text
    else:
        body = resp.text

    return resp.status_code, body, dict(resp.headers), url


## Insurer v1 - test endpoints via DPoP

Below you will find one code cell per **Insurer v1 endpoint**.

Execution order matters:
- Some endpoints require IDs (`CLAIM_ID`, `REPORT_ID`, `DOC_ID`).
- The notebook tries to discover these IDs from previous responses when possible.


In [ ]:
# ---- Setup for Insurer v1 tests ----

import os
import base64
import json

CLAIM_ID = os.getenv("CLAIM_ID") or None
REPORT_ID = os.getenv("REPORT_ID") or None
DOC_ID = os.getenv("DOC_ID") or None


def require(name: str, value: str | None) -> str:
    if not value:
        raise RuntimeError(
            f"{name} is not set. Set {name} (e.g. in this notebook cell or as an environment variable)."
        )
    return value


def set_if_none(name: str, current: str | None, candidate: str | None) -> str | None:
    if current:
        return current
    if candidate:
        print(f"Setting {name} = {candidate}")
        return candidate
    return current


def print_result(status: int, url_used: str, body):
    print("HTTP", status)
    print("URL:", url_used)
    if isinstance(body, (dict, list)):
        print(json.dumps(body, indent=2, ensure_ascii=False))
    else:
        print(body)


# Small allowed example document (text/plain) for upload tests
REPORT_DOC_CONTENT_B64 = base64.b64encode(
    b"Hello from Claimity Partner API demo (Insurer document upload)"
).decode("ascii")

UPLOAD_DOC_REQUEST = {
    "fileName": "insurer-document.txt",
    "contentType": "text/plain",
    "contentBase64": REPORT_DOC_CONTENT_B64,
}

# Example payload for claim creation (vehicle)
CREATE_CLAIM_REQUEST = {
    "Category": "vehicle",
    "PayloadJson": "{}",
}


In [ ]:
# GET /v1/insurers/claims

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path='/v1/insurers/claims',
    access_token=ACCESS_TOKEN,
    params={'page': 1, 'size': 20, 'category': 'vehicle'},
)

print_result(status, url_used, body)

# Best-effort: derive CLAIM_ID from first item
if isinstance(body, dict):
    items = body.get('items') or body.get('Items') or []
    if items:
        first = items[0] or {}
        CLAIM_ID = set_if_none('CLAIM_ID', CLAIM_ID, first.get('id') or first.get('Id'))


In [ ]:
# GET /v1/insurers/claims/{claimId}

claim_id = require('CLAIM_ID', CLAIM_ID)

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}',
    access_token=ACCESS_TOKEN,
)

print_result(status, url_used, body)


In [ ]:
# GET /v1/insurers/claims/{claimId}/documents

claim_id = require('CLAIM_ID', CLAIM_ID)

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}/documents',
    access_token=ACCESS_TOKEN,
)

print_result(status, url_used, body)

# Best-effort: derive CLAIM_ID from first item
if isinstance(body, dict):
    items = body.get('items') or body.get('Items') or []
    if items:
        first = items[0] or {}
        DOC_ID = set_if_none('DOC_ID', DOC_ID, first.get('id') or first.get('Id'))


In [ ]:
# GET /v1/insurers/claims/{claimId}/documents/{documentId}

claim_id = require('CLAIM_ID', CLAIM_ID)
document_id = require('DOC_ID', DOC_ID)

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}/documents/{document_id}',
    access_token=ACCESS_TOKEN,
)

print_result(status, url_used, body)


In [ ]:
# POST /v1/insurers/claims/{claimId}/documents

claim_id = require('CLAIM_ID', CLAIM_ID)

status, body, headers, url_used = send_with_dpop(
    method='POST',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}/documents',
    access_token=ACCESS_TOKEN,
    payload_json=UPLOAD_DOC_REQUEST,
)

print_result(status, url_used, body)

# Best-effort: derive DOC_ID from response
if isinstance(body, dict):
    DOC_ID = set_if_none('DOC_ID', DOC_ID, body.get('id') or body.get('Id'))


In [ ]:
# GET /v1/insurers/claims/{claimId}/reports

claim_id = require('CLAIM_ID', CLAIM_ID)

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}/reports',
    access_token=ACCESS_TOKEN,
)

print_result(status, url_used, body)

# Best-effort: derive REPORT_ID from response
if isinstance(body, dict):
    items = body.get('items') or body.get('Items') or []
    if items:
        first = items[0] or {}
        REPORT_ID = set_if_none('REPORT_ID', REPORT_ID, first.get('id') or first.get('Id'))


In [ ]:
# GET /v1/insurers/claims/{claimId}/reports/{submissionId}/documents

claim_id = require('CLAIM_ID', CLAIM_ID)
report_id = require('REPORT_ID', REPORT_ID)

status, body, headers, url_used = send_with_dpop(
    method='GET',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims/{claim_id}/reports/{report_id}/documents',
    access_token=ACCESS_TOKEN,
)

print_result(status, url_used, body)


In [ ]:
# POST /v1/insurers/claims:validate

status, body, headers, url_used = send_with_dpop(
    method='POST',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims:validate',
    access_token=ACCESS_TOKEN,
    payload_json=CREATE_CLAIM_REQUEST,
)

print_result(status, url_used, body)


In [ ]:
# POST /v1/insurers/claims

status, body, headers, url_used = send_with_dpop(
    method='POST',
    base_url=API_BASE_URL,
    path=f'/v1/insurers/claims',
    access_token=ACCESS_TOKEN,
    payload_json=CREATE_CLAIM_REQUEST,
)

print_result(status, url_used, body)
